In [2]:
from lingua import Language, LanguageDetectorBuilder
import pandas as pd
import re

# Cleaning up User and Character Data to Look at in R

In [3]:
headers = ["username",
"display name",
"num_followers",
"num_following",
"num_interactions",
"bio",
"num_characters"
          ]
df_data = []
network = []

with open("/data/characterai/users.jsonl") as inf:
    for line in inf:
        dat = eval(line.replace(", null,", ", None,"))
        df_data.append(dat[:-2] + [len(dat[-2])])
        network += [[dat[0],x] for x in dat[-1]]
    
df = pd.DataFrame(df_data,columns=headers)
df.to_csv("/data/characterai/users_data.csv", index=False)
with open("/data/characterai/net.csv","w") as net_f:
    [net_f.write("\t".join(x)+"\n") for x in network]

In [3]:
headers = ["character url",
"creator",
"interactions",
"likes",
"name",
"title",
"greeting",
"description",
"definition"
]

df_data = [eval(re.sub(", null", ", None", line)) 
               for line in open("/data/characterai/characters_5per.jsonl")]
df = pd.DataFrame(df_data,columns=headers)

df.to_parquet(
    "/data/characterai/character_data_5per_all_col.parquet",index=False)

# df[['character url','creator','interactions','likes','name','title','greeting']].to_parquet(
#     "/data/characterai/character_data.parquet",index=False)

In [8]:
for col in ['title','greeting','description','definition']:
    print(f"{col}: {1-df[col].isnull().sum()/len(df)}")

title: 1.0
greeting: 1.0
description: 0.33127444026118114
definition: 0.038941055670471236


# Language Detection

In [2]:

headers = [
    "character_url",
    "creator",
    "interactions",
    "likes",
    "name",
    "title",
    "greeting",
    "description",
    "definition",
]
df_data = [eval(re.sub(", null", ", None", line)) 
           for line in open("/data/characterai/characters_5per.jsonl", 
                            encoding="utf-8").readlines()]
df = pd.DataFrame(df_data, columns=headers)
print("Detecting Language")


to_detect = []
for i, row in tqdm(df.iterrows(),total=len(df)):
    if len(row.greeting) > 0:
        to_detect.append([row.character_url,row.greeting])

l = LanguageDetectorBuilder.from_all_languages().build()

res = l.detect_languages_in_parallel_of([x[1] for x in to_detect])

res_df = pd.DataFrame([(to_detect[i][0],
                   to_detect[i][1],
                   res[i]) for i in range(len(to_detect))],
                  columns = ['url','greeting','lang'])

res_df[res_df.lang == Language.ENGLISH][['url','greeting']].to_parquet("/data/characterai/english_5per.parquet")

Detecting Language


100%|████████████████████████████████████████████████████████████████████████████████████| 3023955/3023955 [02:19<00:00, 21663.72it/s]


In [25]:
res_df['lang'] = res_df.lang.apply(lambda x : x.name if x else '')

In [26]:
res_df.to_parquet("/data/characterai/full_language_tagged_greetings.parquet")

In [58]:
r_eng = res_df[res_df.lang == 'ENGLISH'][['url','greeting']]

In [59]:
r_eng['str_len'] = r_eng.greeting.str.len()

In [60]:
r_eng = r_eng[r_eng.str_len > 40]

In [61]:
len(r_eng[r_eng.greeting.str.contains("{{user}}")|r_eng.greeting.str.contains("you") ]), len(r_eng)

(1880405, 2193955)

In [65]:
sp = r_eng[r_eng.greeting.str.contains("{{user}}")|r_eng.greeting.str.lower().str.contains("you") ]

# Write out Named Entity Data from CCR

In [10]:
df = pd.read_parquet("/data/characterai/updated/mask_result.parquet")

import re
def clean_text(text):
    # Remove possessive 's (e.g., John's -> John)
    text = re.sub(r"'s\b", "", text)
    # Remove punctuation (everything except word characters and whitespace)
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()
    
rz = []
n = 0
for i, row in df.iterrows():
    if len(row.greeting) < 50:
        continue
    n +=1
    url = row.url.replace("https://character.ai/chat/","")
    clean_ents = {clean_text(e) for e in row.ents}
    for phrase in clean_ents:
        if phrase.endswith("s") and phrase[:-1] in clean_ents:
            continue  # Skip the plural form if singular exists
        rz.append([url, phrase])


pd.DataFrame(rz, columns=['url','entity']).to_csv("/data/characterai/updated/ents.csv",index=False)
